# GISLR · Stage 1 — `BiLSTM` architecture benchmark

**Pipeline stage** for GISLR (TODO §4 architecture benchmarking): per-subset feature caches (shared with `gislr.1.model.gru.ipynb`) → BiLSTM training run(s) → results & run-folder docs. **⚠ OFFLINE-ONLY accuracy reference** — the backward pass reads future frames; this model can **never** be a deployment candidate (project constraint).

**Why:** it prices how much accuracy streaming causality costs. Readout: forward state at the last valid frame concatenated with the backward state at t=0 (which has seen the whole sequence) → 2×hidden head. Compare its val accuracy against the unidirectional GRU/LSTM to quantify the causality gap; nothing from this notebook feeds the deployment path, and there is no export section.

Built **2026-07-17** as a sibling of `gislr.1.model.gru.ipynb` (regime **v2-plateau-300**, TODO §3.2): identical canonical split, feature caches, training driver (auto-resume + early stopping on val-acc plateau + fresh timestamped folder per training start) and run-doc/`metadata.json` emission. The only differences are the §1 architecture-identity block, `TRAIN_SUBSETS`, and the §2.2 model class. Default subset: **ME_126** (best canonical-eval subset; `FP_118` may displace it once its pending eval runs — swap `TRAIN_SUBSETS` accordingly).

**Artifacts produced**

| artifact | path |
|---|---|
| per-subset feature caches (skip-if-exists, **shared with the other architecture notebooks**) | `cache/{train,val}_<tag>_{data,offsets}.npy` |
| run-dir pointer per subset (drives auto-resume) | `cache/bilstm_runs/<tag>.txt` |
| checkpoints (gitignored) | `models/gislr/bilstm/<timestamp>/bilstm_{latest,best}.pt` |
| run docs (the committed record) | `models/gislr/bilstm/<timestamp>/{README.md,data.md,metadata.json}` |
| landmark indices + hyperparams | `models/gislr/bilstm/<timestamp>/cache/{landmarks.npy,hyp.json}` |
| learning curves | `models/gislr/bilstm/<timestamp>/assets/learning_loss_accuracy.png` |

**Resumability** — cache builds skip existing files and write atomically; training auto-resumes from `bilstm_latest.pt` (saved every epoch, atomically, incl. the early-stopping counter and LR-scheduler state). A **finished** run (early-stopped or epoch cap) is never reused — re-running §5 starts a fresh timestamped run. An interrupt never forces a full rerun.

**Evaluation** — §7 writes each run's docs + `metadata.json` (aggregated into `src/models/index.csv` by `scripts/build_model_index.py`) and prints the canonical per-class eval command (`scripts/eval_gru.py`, which dispatches on the checkpoint's `arch` key) — **run by the user, after training**. Leaderboard comparison uses the same stratified 90/10 seed-42 split as every other run.

**Kernel requirements**: project `.venv`, CWD = `src/`, CUDA GPU. `ME_126`'s feature cache (~5.6 GB) is reused if the GRU notebook already built it.

## 1. Setup

All imports + every shared tunable. **`TRAIN_SUBSETS` is the primary knob** — the §4–§7 cells each re-declare their own working list from it, so tweaking one section never requires re-running another. `COORDS = "xy"` is the future z-drop ablation (TODO §3.1) and flows through cache tags, checkpoints and run docs automatically.

In [ ]:
# ============================================================
# Setup — imports, tunables, resolved paths
# ============================================================
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

from modules.dataset.landmark.subsets import get_subset

SEED = 42            # canonical project seed (split + training)
ROWS_PER_FRAME = 543  # holistic rows per frame (GISLR parquet layout)
MAX_SEQ_LEN = 128    # uniform-subsample cap, identical to the baseline
COORDS = "xyz"       # coordinate channels fed to the model ("xy" drops z — TODO §3.1)

# architecture identity — flows through run folders, checkpoints, run docs,
# metadata.json and src/models/index.csv. The sibling architecture notebooks
# (lstm / bilstm / cnn1d) change ONLY this block, TRAIN_SUBSETS and the model
# class in §2.2 — everything else is byte-identical.
ARCH = "bilstm"                     # models/gislr/<ARCH>/, cache/<ARCH>_runs/, <ARCH>_best.pt
MODEL_NAME = "BiLSTM"
STREAMING = False                 # False only for offline references (BiLSTM)
ARCH_DESC = "bidirectional LSTM, fwd-last + bwd-first readout, OFFLINE-ONLY reference"

#: benchmark on the best canonical-eval subset for GRU comparability (TODO §4)
#: — THE knob of this notebook. FP_118 may displace ME_126 once its pending
#: canonical eval runs; swap accordingly.
TRAIN_SUBSETS = ["ME_126"]

# training regime v2 (TODO §3.2, 2026-07-17): bigger batch, epoch cap 300 with
# early stopping on val-accuracy plateau, ReduceLROnPlateau. OneCycleLR was
# dropped deliberately — it anneals over a fixed total epoch count, which
# early stopping would truncate mid-anneal. lr scaled up with the batch size.
# v2 runs are NOT hyperparameter-comparable to the v1-onecycle-60 family
# (batch 192 / lr 3e-4 / 60 epochs); metadata.json records the regime per run.
TRAIN_REGIME = "v2-plateau-300"
HYP = dict(batch_size=512, lr=1e-3, hidden_size=256, num_layers=2,
           dropout=0.3, weight_decay=1e-4, epochs=300, grad_clip=5.0,
           lr_factor=0.5, lr_patience=5,       # ReduceLROnPlateau on val acc
           es_patience=15, es_min_delta=1e-3)  # early stop: no +0.1pt val-acc gain in 15 epochs

CACHE_DIR = Path("cache")                  # per-subset feature caches (gitignored, shared across architectures)
RUN_PTR_DIR = CACHE_DIR / f"{ARCH}_runs"   # <tag>.txt -> active run dir (auto-resume)
MODELS_ROOT = Path(f"models/gislr/{ARCH}")
CKPT_LATEST, CKPT_BEST = f"{ARCH}_latest.pt", f"{ARCH}_best.pt"
CACHE_DIR.mkdir(exist_ok=True)
RUN_PTR_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "training requires the CUDA build of torch (uv sync)"
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

# download only the GISLR competition data (never `import modules.paths` —
# that resolves every dataset incl. POPSIGN at import time)
DATA_DIR = Path(kagglehub.competition_download("asl-signs"))
sign2idx = json.loads((DATA_DIR / "sign_to_prediction_index_map.json").read_text())
NUM_CLASSES = len(sign2idx)


def subset_tag(name: str, coords: str = COORDS) -> str:
    """Cache/run tag: 'ME_126' -> 'me126' (+'-xy' when z is dropped) — matches
    the naming of the existing ME-126 caches so they are reused, not rebuilt."""
    tag = name.lower().replace("_", "")
    return tag if coords == "xyz" else f"{tag}-{coords}"


print(torch.__version__, torch.cuda.get_device_name(0))
print(f"DATA_DIR    = {DATA_DIR}")
print(f"CACHE_DIR   = {CACHE_DIR.resolve()}")
print(f"MODELS_ROOT = {MODELS_ROOT.resolve()} · {MODEL_NAME} · regime {TRAIN_REGIME}")
print(f"{NUM_CLASSES} classes · subsets to train: " + ", ".join(
    f"{n} ({len(get_subset(n))} lm, probe {get_subset(n).probe_acc_global:.4f})"
    for n in TRAIN_SUBSETS))

## 2. Reusable core

Run-once definitions (functions/classes only, no mutable state): **2.1** canonical split + feature-cache builder · **2.2** dataset / collate / model · **2.3** training driver with auto-resume. Together with §1 these are the only cells later sections depend on — every §3+ cell is independently re-runnable and loads its inputs from disk, not from other cells' live memory.

In [ ]:
# ============================================================
# 2.1 Reusable core — canonical split + per-subset feature caches
# ============================================================
def get_canonical_split() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Stratified 90/10 split, random_state=42 — identical to every leaderboard
    run and to scripts/eval_gru.py (the canonical evaluation). Deterministic and
    cheap, so consumer cells call it themselves instead of sharing live state."""
    df = pd.read_csv(DATA_DIR / "train.csv")
    missing = set(df["sign"].unique()) - set(sign2idx)
    assert not missing, f"signs missing from the official label map: {missing}"
    df["label"] = df["sign"].map(sign2idx)
    tr, va = train_test_split(df, test_size=0.1, stratify=df["sign"], random_state=SEED)
    return tr.reset_index(drop=True), va.reset_index(drop=True)


def load_video_subset(path, rows: np.ndarray, coords: str = COORDS) -> np.ndarray:
    """One parquet -> (T, len(rows), len(coords)) float32, NaN->0.
    Row selection happens here so caches only ever hold the subset's data."""
    cols = list(coords)
    table = pq.read_table(path, columns=cols)
    data = np.column_stack([table.column(c).to_numpy() for c in cols])
    n = data.shape[0] // ROWS_PER_FRAME
    arr = data.reshape(n, ROWS_PER_FRAME, len(cols))[:, rows, :].astype(np.float32)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def build_subset_cache(df: pd.DataFrame, prefix: str, subset, coords: str = COORDS):
    """Decode every parquet once into one flat float32 array + frame offsets.

    Resumable policy: skipped when both files exist; writes are atomic
    (temp file + os.replace), so an interrupt never leaves a half-written cache.
    """
    tag = subset_tag(subset.name, coords)
    data_path = CACHE_DIR / f"{prefix}_{tag}_data.npy"
    off_path = CACHE_DIR / f"{prefix}_{tag}_offsets.npy"
    if data_path.exists() and off_path.exists():
        print(f"{prefix}/{tag}: cache exists, skipping")
        return data_path, off_path

    t0 = time.time()
    paths = [DATA_DIR / p for p in df["path"]]
    rows = subset.array
    chunks, offsets = [], [0]
    with ThreadPoolExecutor(12) as ex:
        for i, arr in enumerate(ex.map(lambda p: load_video_subset(p, rows, coords), paths)):
            chunks.append(arr.reshape(-1))
            offsets.append(offsets[-1] + arr.shape[0])
            if i % 5000 == 0:
                print(f"  {prefix}/{tag} {i}/{len(paths)} ({time.time()-t0:.0f}s)", flush=True)
    flat = np.concatenate(chunks)
    for target, payload in ((data_path, flat),
                            (off_path, np.asarray(offsets, dtype=np.int64))):
        tmp = target.with_suffix(".tmp.npy")
        np.save(tmp, payload)
        os.replace(tmp, target)
    print(f"{prefix}/{tag}: cached {len(df)} videos, {flat.nbytes/1e9:.2f} GB "
          f"({time.time()-t0:.0f}s)")
    return data_path, off_path

In [ ]:
# ============================================================
# 2.2 Reusable core — in-RAM dataset, collate, BiLSTM
# ============================================================
class SubsetArrayDataset(Dataset):
    """Flat in-RAM cache + offsets; uniform subsample past MAX_SEQ_LEN.

    In-RAM with num_workers=0 is deliberate: it trains GISLR at ~0.3 min/epoch
    and sidesteps Windows spawn-pickling entirely, which is why this class can
    live in the notebook instead of a module.
    """

    def __init__(self, df, data_path, off_path, feature_dim, max_seq_len=MAX_SEQ_LEN):
        self.labels = df["label"].to_numpy()
        self.data = np.load(data_path)   # ~5 GB for an ME-126-sized train split
        self.offsets = np.load(off_path)
        self.feature_dim = feature_dim
        self.max_seq_len = max_seq_len
        assert len(self.labels) == len(self.offsets) - 1, "cache/split mismatch"

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        d = self.feature_dim
        arr = self.data[self.offsets[i] * d : self.offsets[i + 1] * d].reshape(-1, d)
        T = arr.shape[0]
        if T > self.max_seq_len:
            arr = arr[np.linspace(0, T - 1, self.max_seq_len).astype(int)]
            T = self.max_seq_len
        return torch.from_numpy(np.ascontiguousarray(arr)), T, int(self.labels[i])


def collate_fn(batch):
    batch.sort(key=lambda x: x[1], reverse=True)  # enforce_sorted packing
    feats, lengths, labels = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    padded = torch.zeros(len(feats), int(lengths[0]), feats[0].shape[1])
    for i, f in enumerate(feats):
        padded[i, : f.shape[0]] = f
    return padded, lengths, labels


class BiLSTM(nn.Module):
    """Bidirectional LSTM — OFFLINE-ONLY accuracy reference: the backward pass
    reads future frames, so this can NEVER be a deployment candidate (project
    constraint). It exists to price how much accuracy streaming causality
    costs vs the unidirectional models.

    Readout: forward direction at the last valid frame + backward direction at
    t=0 (which has seen the whole sequence), concatenated -> 2*hidden head."""

    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=True)
        self.head = nn.Sequential(nn.LayerNorm(2 * hidden_size), nn.Dropout(dropout),
                                  nn.Linear(2 * hidden_size, num_classes))

    def forward(self, x, lengths):
        x = self.input_norm(x)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=True)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)  # (B, T, 2H)
        H = out.size(-1) // 2
        idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, H).to(out.device)
        fwd_last = out[..., :H].gather(1, idx).squeeze(1)  # fwd state at last valid frame
        bwd_first = out[:, 0, H:]                          # bwd state at t=0 (saw everything)
        return self.head(torch.cat([fwd_last, bwd_first], dim=-1))


def build_model(feature_dim: int) -> nn.Module:
    """Factory used by the §2.3 training driver — the ONLY constructor call,
    so the sibling architecture notebooks swap the model by editing this cell."""
    return BiLSTM(feature_dim, HYP["hidden_size"], HYP["num_layers"],
                  NUM_CLASSES, HYP["dropout"])

In [ ]:
# ============================================================
# 2.3 Reusable core — training driver with auto-resume + early stopping
# ============================================================
def resolve_run_dir(tag: str) -> Path:
    """models/gislr/<ARCH>/<timestamp>/ (timestamp = training start time). The
    pointer file makes re-runs land in the SAME folder only while that run is
    still unfinished (auto-resume after an interrupt). A FINISHED run — early-
    stopped or epoch cap reached — is never reused: every new training gets a
    fresh timestamped folder, even with identical conditions."""
    ptr = RUN_PTR_DIR / f"{tag}.txt"
    if ptr.exists():
        latest = Path(ptr.read_text().strip()) / CKPT_LATEST
        if latest.exists():
            ck = torch.load(latest, map_location="cpu", weights_only=False)
            finished = ck.get("finished", ck["epoch"] + 1 >= ck["hyp"]["epochs"])
            if not finished:
                return latest.parent  # interrupted -> resume in place
    run_dir = MODELS_ROOT / datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    ptr.write_text(str(run_dir))
    return run_dir


def atomic_torch_save(state: dict, path: Path) -> None:
    tmp = path.with_suffix(".pt.tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
    grad_clip=HYP["grad_clip"],
    desc="",
):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        for feats, lengths, labels in tqdm(loader, desc=desc, leave=False):
            feats = feats.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                logits = model(feats, lengths)
                loss = criterion(logits, labels)
            if train_mode:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(-1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def train_subset(subset_name: str, coords: str = COORDS) -> Path:
    """Train one run for `subset_name` under the current regime. Builds missing
    caches and resumes an interrupted run from its checkpoint — including the
    early-stopping counter and LR-scheduler state, so an interrupt never
    changes when the run would have stopped. A finished run is never reused:
    calling this again trains a NEW timestamped run."""
    subset = get_subset(subset_name)
    tag = subset_tag(subset_name, coords)
    feature_dim = len(subset) * len(coords)

    train_split, val_split = get_canonical_split()
    tr_data, tr_off = build_subset_cache(train_split, "train", subset, coords)
    va_data, va_off = build_subset_cache(val_split, "val", subset, coords)

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    train_ds = SubsetArrayDataset(train_split, tr_data, tr_off, feature_dim)
    val_ds = SubsetArrayDataset(val_split, va_data, va_off, feature_dim)
    g = torch.Generator()
    g.manual_seed(SEED)
    train_loader = DataLoader(
        train_ds,
        batch_size=HYP["batch_size"],
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        generator=g,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=HYP["batch_size"],
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    model = build_model(feature_dim).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=HYP["lr"], weight_decay=HYP["weight_decay"]
    )
    # plateau-coupled LR schedule: halves the LR when val acc stalls; the early
    # stop below watches the same signal with a longer patience, so the LR
    # gets lr_patience epochs to rescue a plateau before the run ends
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=HYP["lr_factor"], patience=HYP["lr_patience"]
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda")

    run_dir = resolve_run_dir(tag)
    (run_dir / "cache").mkdir(exist_ok=True)
    (run_dir / "assets").mkdir(exist_ok=True)
    np.save(run_dir / "cache" / "landmarks.npy", subset.array)
    (run_dir / "cache" / "hyp.json").write_text(
        json.dumps(
            {
                **HYP,
                "seed": SEED,
                "max_seq_len": MAX_SEQ_LEN,
                "coords": coords,
                "subset": subset_name,
                "n_landmarks": len(subset),
                "feature_dim": feature_dim,
                "n_params": n_params,
                "num_workers": 0,
                "arch": ARCH,
                "training_regime": TRAIN_REGIME,
                "training_source": f"gislr.1.model.{ARCH}.ipynb",
            },
            indent=2,
        )
    )
    print(f"{tag}: run dir {run_dir} · input dim {feature_dim} · params {n_params:,}")

    latest, best = run_dir / CKPT_LATEST, run_dir / CKPT_BEST
    start_epoch, best_val_acc, epochs_since_gain = 0, 0.0, 0
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }
    if latest.exists():
        ck = torch.load(latest, map_location=device, weights_only=False)
        model.load_state_dict(ck["model_state"])
        optimizer.load_state_dict(ck["optimizer_state"])
        scheduler.load_state_dict(ck["scheduler_state"])
        start_epoch, best_val_acc, history = (
            ck["epoch"] + 1,
            ck["best_val_acc"],
            ck["history"],
        )
        epochs_since_gain = ck.get("epochs_since_gain", 0)
        print(
            f"{tag}: resumed at epoch {start_epoch}, best {best_val_acc:.4f}, "
            f"plateau counter {epochs_since_gain}/{HYP['es_patience']}"
        )

    t0 = time.time()
    for epoch in range(start_epoch, HYP["epochs"]):
        tr_loss, tr_acc = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            desc=f"{tag} ep{epoch + 1} train",
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, desc=f"{tag} ep{epoch + 1} val"
        )
        scheduler.step(val_acc)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])  # one point per epoch
        is_best = val_acc > best_val_acc
        # plateau detection: only a val-acc gain > es_min_delta resets the counter
        # (is_best still saves the best checkpoint on ANY improvement)
        epochs_since_gain = (
            0 if val_acc > best_val_acc + HYP["es_min_delta"] else epochs_since_gain + 1
        )
        best_val_acc = max(best_val_acc, val_acc)
        early_stop = epochs_since_gain >= HYP["es_patience"]
        finished = early_stop or (epoch + 1 >= HYP["epochs"])
        state = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "history": history,
            "sign2idx": sign2idx,
            "hyp": {**HYP, "seed": SEED, "max_seq_len": MAX_SEQ_LEN, "num_workers": 0},
            "feature_dim": feature_dim,
            "landmarks": subset.array.tolist(),
            "subset_name": subset_name,
            "coords": coords,
            "arch": ARCH,
            "training_regime": TRAIN_REGIME,
            "epochs_since_gain": epochs_since_gain,
            "finished": finished,
        }
        atomic_torch_save(state, latest)  # every epoch -> resume-safe
        if is_best:
            atomic_torch_save(state, best)
        print(
            f"{tag} epoch {epoch + 1:03d}/{HYP['epochs']} | tr_loss {tr_loss:.4f} "
            f"acc {tr_acc:.4f} | val_loss {val_loss:.4f} acc {val_acc:.4f}"
            f"{' *BEST*' if is_best else ''} | lr {history['lr'][-1]:.2e} "
            f"| plateau {epochs_since_gain}/{HYP['es_patience']} "
            f"| {(time.time() - t0) / 60:.1f} min",
            flush=True,
        )
        if early_stop:
            print(
                f"{tag}: EARLY STOP at epoch {epoch + 1} — no val-acc gain "
                f"> {HYP['es_min_delta']} for {HYP['es_patience']} epochs"
            )
            break
    print(f"{tag}: DONE best_val_acc={best_val_acc:.4f} run_dir={run_dir}")
    return run_dir

## 3. Canonical split check

Sanity-check the stratified 90/10 seed-42 split that every leaderboard run and `scripts/eval_gru.py` share (85,029 train / 9,448 val). Purely diagnostic — the training driver calls `get_canonical_split()` itself.

In [ ]:
# ============================================================
# Canonical split sanity check — must match the leaderboard split exactly
# ============================================================
train_split, val_split = get_canonical_split()
assert len(val_split) == 9448, "val-set size drifted — leaderboard comparability broken"
print(f"train {len(train_split)} / val {len(val_split)} · "
      f"{train_split['sign'].nunique()} classes · "
      f"val classes {val_split['sign'].nunique()}")
display(train_split.head(3))

## 4. Feature caches (per subset)

One flat float32 array + offsets index per (split, subset), decoded from raw parquet once and reused by every later run (skip-if-exists, atomic writes). `ME_126`'s cache from the `20260715-190729` run is picked up unchanged thanks to the shared `me126` tag. Delete a file pair in `cache/` to force a rebuild.

In [ ]:
# ============================================================
# Build per-subset feature caches (tunable: CACHE_SUBSETS)
# Safe to re-run any time — existing caches are skipped.
# ============================================================
CACHE_SUBSETS = TRAIN_SUBSETS

train_split, val_split = get_canonical_split()
for name in CACHE_SUBSETS:
    subset = get_subset(name)
    build_subset_cache(train_split, "train", subset)
    build_subset_cache(val_split, "val", subset)

total_gb = sum(f.stat().st_size for f in CACHE_DIR.glob("*_data.npy")) / 1e9
print(f"\ntotal feature-cache size on disk: {total_gb:.1f} GB")

## 5. Training runs (TODO §4, regime §3.2)

One run folder per subset under `models/gislr/bilstm/<timestamp>/` (timestamp = training start), trained under regime **v2-plateau-300**: batch 512, lr 1e-3, ReduceLROnPlateau, epoch cap 300 with early stopping on val-acc plateau. **Auto-resume**: interrupting and re-running this cell continues each unfinished subset from its last epoch (early-stopping counter and LR-scheduler state included). **Finished runs are never reused** — re-running this cell trains a fresh timestamped run per subset, so trim `RUN_SUBSETS` to what you actually want to (re)train. Per-class evaluation is *not* done here — that is §7's handoff to `scripts/eval_gru.py`.

In [ ]:
# ============================================================
# Train one BiLSTM run per subset under the current regime (tunable:
# RUN_SUBSETS). Runs stop early on val-accuracy plateau (HYP es_*).
# Resume-safe: re-run after any interrupt resumes the unfinished run.
# NOTE: finished runs are NOT skipped — re-running this cell trains a
# fresh run (new timestamped folder) per subset, even with identical
# conditions. Trim RUN_SUBSETS if you don't want that.
# ============================================================
RUN_SUBSETS = TRAIN_SUBSETS

run_dirs = {}
for name in RUN_SUBSETS:
    run_dirs[name] = train_subset(name)

print("\nrun folders:")
for name, rd in run_dirs.items():
    print(f"  {name}: {rd}")

## 6. Learning curves & comparison

Reads each subset's checkpoint **from disk** (via the run pointers), so it works in a fresh kernel without re-running §5. Saves each run's curves into its `assets/` folder and tabulates best val accuracy against the two committed references.

In [ ]:
# ============================================================
# Learning curves + comparison table (tunable: REPORT_SUBSETS)
# Loads the latest checkpoint per run pointer — no dependency on §5 live state.
# ============================================================
REPORT_SUBSETS = TRAIN_SUBSETS
BASELINES = {
    "GRU FULL_543 (20260713-213000, v1 regime)": 0.7059,
    "GRU ME_126 (20260715-190729, v1 regime)": 0.7373,
}

rows = []
for name in REPORT_SUBSETS:
    ptr = RUN_PTR_DIR / f"{subset_tag(name)}.txt"
    if not ptr.exists():
        print(f"{name}: no run pointer yet — train it in §5 first")
        continue
    run_dir = Path(ptr.read_text().strip())
    ck = torch.load(run_dir / CKPT_LATEST, map_location="cpu", weights_only=False)
    h = ck["history"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(h["train_loss"], label="train")
    axes[0].plot(h["val_loss"], label="val")
    axes[0].set_title(f"{name} — loss")
    axes[0].legend()
    axes[1].plot(h["train_acc"], label="train")
    axes[1].plot(h["val_acc"], label="val")
    axes[1].set_title(f"{name} — accuracy")
    axes[1].legend()
    axes[2].plot(h["lr"])
    axes[2].set_title("LR schedule")
    fig.tight_layout()
    fig.savefig(run_dir / "assets" / "learning_loss_accuracy.png", dpi=110)
    plt.show()

    rows.append(
        dict(
            subset=name,
            run=run_dir.name,
            n_landmarks=len(get_subset(name)),
            feature_dim=ck["feature_dim"],
            epochs=len(h["val_acc"]),
            best_val_acc=round(ck["best_val_acc"], 4),
            probe_acc=get_subset(name).probe_acc_global,
        )
    )

if rows:
    display(
        pd.DataFrame(rows)
        .sort_values("best_val_acc", ascending=False)
        .reset_index(drop=True)
    )
for label, acc in BASELINES.items():
    print(f"reference — {label}: {acc:.4f}")

## 7. Run documentation + canonical per-class eval handoff

Auto-generates each run's `data.md` and a `README.md` **draft** (both required by the registry convention), then prints the exact `scripts/eval_gru.py` command per run — **run those from the repo root yourself**; this notebook never evaluates. Drafts are overwritten on re-run; finalize a README (per-class analysis, leaderboard displacement, `src/models/README.md` update, dated `docs/` report) only after its eval has run.

In [ ]:
# ============================================================
# Write data.md + README.md draft + metadata.json per run; print eval commands
# (tunable: DOC_SUBSETS). Reads everything from disk.
# ============================================================
DOC_SUBSETS = TRAIN_SUBSETS

eval_cmds = []
for name in DOC_SUBSETS:
    ptr = RUN_PTR_DIR / f"{subset_tag(name)}.txt"
    if not ptr.exists():
        print(f"{name}: no run yet — skipping")
        continue
    run_dir = Path(ptr.read_text().strip())
    ck = torch.load(run_dir / CKPT_LATEST, map_location="cpu", weights_only=False)
    meta = json.loads((run_dir / "cache" / "hyp.json").read_text())
    subset = get_subset(name)
    epochs_done = len(ck["history"]["val_acc"])
    complete = ck.get("finished", epochs_done >= ck["hyp"]["epochs"])
    early_stopped = bool(complete and epochs_done < ck["hyp"]["epochs"])
    coords = ck.get("coords", "xyz")
    rd = run_dir.as_posix()

    (run_dir / "data.md").write_text(f"""# Data — GISLR · {name} subset

- **dataset**: Kaggle `asl-signs` (GISLR), 94,477 videos, 250 classes
- **landmark subset**: **{name}** ({len(subset)} landmarks) from the canonical
  registry `src/modules/dataset/landmark/subsets.py`; exact indices in
  [`cache/landmarks.npy`](cache/landmarks.npy)
- **channels**: {coords} -> feature_dim {ck["feature_dim"]}
- **preprocessing**: NaN->0, uniform subsample to <= {MAX_SEQ_LEN} frames
  (identical to the 20260713-213000 baseline)
- **split**: stratified 90/10, `random_state={SEED}` -> 85,029 train /
  9,448 val (the canonical leaderboard split)
- **feature caches**: `src/cache/{{train,val}}_{subset_tag(name, coords)}_{{data,offsets}}.npy`
  (shared across the architecture notebooks)
- **subset provenance**: {subset.provenance}
""", encoding="utf-8")

    status_note = "" if complete else " — **TRAINING INCOMPLETE**"
    stop_txt = ", early-stopped on plateau" if early_stopped else ""
    eval_cmd = (f".venv/Scripts/python.exe scripts/eval_gru.py "
                f"src/{rd}/{CKPT_BEST} src/{rd} "
                f"--landmarks src/{rd}/cache/landmarks.npy")
    (run_dir / "README.md").write_text(f"""# {MODEL_NAME} — {name} subset ({run_dir.name})

> Draft auto-generated by `gislr.1.model.{ARCH}.ipynb` (section 7){status_note}.
> Finalize after running the canonical per-class eval below.

## Training conditions

- architecture: {MODEL_NAME} ({ARCH_DESC}, hidden {HYP["hidden_size"]} x {HYP["num_layers"]} layers,
  dropout {HYP["dropout"]}) — {meta["n_params"]:,} params{"" if STREAMING else "  ·  **OFFLINE-ONLY accuracy reference — never a deployment candidate**"}
- input: {name} ({len(subset)} landmarks) x {coords} = {ck["feature_dim"]} features/frame
- training regime **{TRAIN_REGIME}**: batch {HYP["batch_size"]}, lr {HYP["lr"]}
  (ReduceLROnPlateau: factor {HYP["lr_factor"]}, patience {HYP["lr_patience"]}),
  weight decay {HYP["weight_decay"]}, epoch cap {HYP["epochs"]} with early
  stopping (patience {HYP["es_patience"]}, min-delta {HYP["es_min_delta"]}),
  grad-clip {HYP["grad_clip"]}, CE + label smoothing 0.1, AMP, seed {SEED},
  max_seq_len {MAX_SEQ_LEN}, num_workers 0 (in-RAM cache)
- NOTE: {TRAIN_REGIME} runs are not hyperparameter-comparable to the
  v1-onecycle-60 family (batch 192, lr 3e-4, 60 epochs OneCycleLR — all runs
  before 2026-07-17); the canonical eval split/metric below is unchanged and
  remains the leaderboard requirement.
- data: see [data.md](data.md)

## Results (training loop)

- best val accuracy: **{ck["best_val_acc"]:.4f}** ({epochs_done}/{HYP["epochs"]} epochs{stop_txt})
- GRU references (v1 regime): FULL_543 baseline 70.59% (20260713-213000) · ME_126 73.73% (20260715-190729)

![learning curves](assets/learning_loss_accuracy.png)

## Canonical per-class evaluation — PENDING

Run from the repo root:

```bash
{eval_cmd}
```

Then record overall/macro accuracy + per-class artifacts here and update the
leaderboard in `src/models/README.md` (displacement requires this exact
split/metric).
""", encoding="utf-8")

    # metadata.json — the machine-readable run record aggregated into
    # src/models/index.csv by scripts/build_model_index.py. Canonical-eval
    # fields written by scripts/eval_gru.py survive notebook re-runs.
    meta_path = run_dir / "metadata.json"
    prev = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    best_epoch = int(np.argmax(ck["history"]["val_acc"])) + 1 if complete else None
    metadata = {
        "schema_version": 1,
        "run_id": run_dir.name,
        "dataset": "gislr",
        "architecture": ARCH,
        "model_name": MODEL_NAME,
        "streaming": STREAMING,
        "subset": name,
        "n_landmarks": len(subset),
        "coords": coords,
        "feature_dim": int(ck["feature_dim"]),
        "n_params": meta["n_params"],
        "n_classes": NUM_CLASSES,
        "split": f"stratified 90/10, random_state={SEED}",
        "n_val": 9448,
        "trained_date": f"{run_dir.name[:4]}-{run_dir.name[4:6]}-{run_dir.name[6:8]}",
        "training_source": f"gislr.1.model.{ARCH}.ipynb",
        "training_regime": TRAIN_REGIME,
        "epochs": epochs_done,
        "best_epoch": best_epoch,
        "early_stopped": early_stopped,
        "train_val_acc": round(float(ck["best_val_acc"]), 4),
        "eval_status": prev.get("eval_status", "pending"),
        "overall_accuracy": prev.get("overall_accuracy"),
        "macro_accuracy": prev.get("macro_accuracy"),
        "median_class_accuracy": prev.get("median_class_accuracy"),
        "n_classes_below_50pct": prev.get("n_classes_below_50pct"),
        "hyperparameters": {
            "batch_size": HYP["batch_size"], "lr": HYP["lr"],
            "lr_schedule": f"ReduceLROnPlateau(factor={HYP['lr_factor']}, patience={HYP['lr_patience']})",
            "early_stopping": f"patience={HYP['es_patience']}, min_delta={HYP['es_min_delta']}",
            "epoch_cap": HYP["epochs"],
            "loss": "CE + label smoothing 0.1",
            "precision": "AMP", "hidden_size": HYP["hidden_size"],
            "num_layers": HYP["num_layers"], "dropout": HYP["dropout"],
            "weight_decay": HYP["weight_decay"], "grad_clip": HYP["grad_clip"],
            "seed": SEED, "max_seq_len": MAX_SEQ_LEN, "num_workers": 0,
        },
        "notes": prev.get("notes",
                          f"{name} subset · {MODEL_NAME} · regime {TRAIN_REGIME}."),
    }
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    eval_cmds.append((name, complete, eval_cmd))
    print(f"{name}: wrote {run_dir / 'data.md'}, README.md draft + metadata.json "
          f"({'complete' if complete else f'{epochs_done}/{HYP['epochs']} epochs'})")

print("\n=== canonical per-class eval — run these from the repo root (user) ===")
for name, complete, cmd in eval_cmds:
    print(f"\n# {name}{'' if complete else '  (WAIT: training incomplete)'}\n{cmd}")
print("\nafterwards: rebuild the index (scripts/build_model_index.py), update")
print("src/models/README.md leaderboard + file a dated docs/ report")

## 8. Export — not applicable

BiLSTM is an **offline-only accuracy reference** (the backward pass reads future frames) and can never be deployed to the streaming pipeline; no export chain will be added to this notebook.